# Genotype and Phenotype Data


- Which dogs belong to the case and control groups used in the study?
- Prepared genotype and phenotype checks examine 164 dogs: 116 GSP and 48 LR, grouped under Scheme A and Scheme B.
- These checks confirm cohort counts, accepted phenotype codes, unique identifiers, and recorded file integrity.
- They do not assess genotype quality, genetic relatedness, or association with training outcomes.


### (Initialization) Import Directory-Free Helpers


In [1]:
# Locate the repository and load shared helpers for reading and checking the prepared study data.

from pathlib import Path
import importlib
import sys

study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/notebook_support.py").exists())
sys.path.insert(0, str(study_root / "scripts"))

read_json = importlib.import_module("notebook_support").read_json
study_path = importlib.import_module("notebook_support").study_path
verify_datasets = importlib.import_module("notebook_support").verify_datasets
require_saved = importlib.import_module("notebook_review").require_saved
run_script = importlib.import_module("notebook_support").run_script

### Case and Control Groups

- Behaviorally eliminated dogs are cases in both schemes.
- Scheme A uses Validated or Certified dogs with no elimination reason as controls.
- Scheme B adds dogs eliminated for Medical, Environmental, or Inconsistency reasons.

| Scheme | Breed | Controls | Behavioral Cases | Total |
| :---: | :---: | :---: | :---: | :---: |
| A | GSP | 96 | 9 | 105 |
| B | GSP | 107 | 9 | 116 |
| A | LR | 39 | 7 | 46 |
| B | LR | 41 | 7 | 48 |


### Checking the Prepared Data

- The prepared data contain 229,991 measured markers before filtering.
- Checks cover unique study identifiers, cohort counts and file integrity.
- Unresolved outcomes are excluded; individual records remain private.


### (Parse) Load Genotype and Phenotype Data


In [2]:
# Verify the prepared genotype files against their recorded hashes before using their cohort information.

inputs = read_json(study_root / "data/01-input/manifest.private.json")
verify_datasets(study_root, inputs)

print("Prepared cohort version:", inputs["version"])
print("Verified genotype datasets:", len(inputs["datasets"]))

Prepared cohort version: v20261002-certified
Verified genotype datasets: 4


### (Consistency Check) Count Cases and Controls Within Each Breed


In [3]:
# Count cases and controls directly from PLINK sample files and compare them with the saved cohort counts.
# Unique identifiers and accepted phenotype codes are checked without displaying individual dog records.

from collections import Counter

def cohort_counts(prefix):
    rows = [line.split() for line in prefix.with_suffix(".fam").read_text(encoding="utf-8").splitlines() if line.strip()]
    assert len({row[1] for row in rows}) == len(rows)
    phenotypes = Counter(row[5] for row in rows)
    assert set(phenotypes) <= {"1", "2"}
    return {"dogs": len(rows), "controls": phenotypes["1"], "cases": phenotypes["2"]}

for dataset in inputs["datasets"]:
    prefix = next(study_path(study_root, p) for p in dataset["files"] if p.endswith(".fam")).with_suffix("")
    counts = cohort_counts(prefix)
    assert all(counts[key] == dataset[key] for key in counts)
    print(dataset["scenario"], dataset["breed"], counts)

success-vs-behelim GSP {'dogs': 105, 'controls': 96, 'cases': 9}
success-vs-behelim LR {'dogs': 46, 'controls': 39, 'cases': 7}
not-behelim-vs-behelim GSP {'dogs': 116, 'controls': 107, 'cases': 9}
not-behelim-vs-behelim LR {'dogs': 48, 'controls': 41, 'cases': 7}
